# 10 — Адаптивный fill по классу/пути дампа

iter08: фикс-расписание (транш каждые 3 мин, ~30-мин окно) грузит нас у ВЕРХА в
затяжном каскаде. Идея: пусть путь дампа диктует темп добора.

**Три схемы (все lookahead-free, полная вселенная −7%/15m):**
- **A. Time-fixed** (база iter02): транш каждые MIN_GAP=3 мин, рамп, ≤N_MAX=10.
- **B. Price-step**: транш, когда close ≤ last_fill·(1−Δ), Δ=2%, рамп, ≤10.
  Быстрый flash → ступени часто (у дна); затяжной → транши идут вниз за ценой.
- **C. Class-conditioned**: классифицируем дамп в триггере по «резкости»
  sharp = r5/r15 (доля 15-мин падения за последние 5 мин). Резкий → time;
  затяжной (grind) → price-step.

При фикс. нотионале более низкий ср.вход = меньше убыток и в норме, и в хвосте.

In [1]:
import sys; sys.path.insert(0,".")
from _lab import *
import numpy as np, pandas as pd
from pathlib import Path
import heapq
WIN=15;H=240;COOLDOWN=10;MIN_GAP=3;N_MAX=10;FEE=0.00075;THR=0.07;DELTA=0.02

def clusters(c,thr):
    if len(c)<WIN+5: return []
    r=np.full(len(c),np.nan); r[WIN:]=c[WIN:]/c[:-WIN]-1
    cond=r<=-thr; out=[]; i=WIN; n=len(c)
    while i<n:
        if not cond[i] or (i>0 and cond[i-1]): i+=1; continue
        start=i;last_true=i;falses=0;j=i+1
        while j<n:
            if cond[j]: last_true=j;falses=0
            else:
                falses+=1
                if falses>=COOLDOWN: break
            j+=1
        out.append((start,last_true)); i=last_true+COOLDOWN+1
    return out

def fills_time(c,st,en):
    pos=[st]; last=st; t=st+1
    while t<=en and len(pos)<N_MAX:
        if t-last>=MIN_GAP: pos.append(t); last=t
        t+=1
    return pos
def fills_price(c,st,en,delta=DELTA):
    pos=[st]; lastc=c[st]; t=st+1
    while t<=en and len(pos)<N_MAX:
        if c[t]<=lastc*(1-delta): pos.append(t); lastc=c[t]
        t+=1
    return pos
def sharpness(c,st):
    r5=c[st]/c[st-5]-1; r15=c[st]/c[st-15]-1
    return r5/r15 if r15!=0 else np.nan   # fraction of 15m drop in last 5m

def avg_entry(c,pos,n):
    pos=[p for p in pos if p+1<n]
    prices=np.array([c[p+1] for p in pos])
    ws=np.arange(1,len(prices)+1,dtype=float); ws/=ws.sum()
    return (ws*prices).sum(), len(prices)

In [2]:
# full-universe pass: pnl per scheme + sharpness + entry depth
rows=[]
for s in list_symbols():
    try: df=ohlcv(s,"2024-01-01","2026-07-01","1min")
    except Exception: continue
    c=df["close"].to_numpy("float64"); ts=df.index; n=len(c)
    for (st,en) in clusters(c,THR):
        ex=en+1+H
        if st<WIN or st+1>=n or ex>=n: continue
        exitp=c[ex]
        shp=sharpness(c,st)
        aT,kT=avg_entry(c,fills_time(c,st,en),n)
        aP,kP=avg_entry(c,fills_price(c,st,en),n)
        # class-conditioned decided AFTER we know sharp threshold; store both, pick later
        rows.append(dict(sym=s,entry=ts[st+1],exit_ts=ts[ex],sharp=shp,
            pnl_time=exitp/aT-1-2*FEE, pnl_price=exitp/aP-1-2*FEE,
            kT=kT,kP=kP, dep_time=aT/c[st+1]-1, dep_price=aP/c[st+1]-1,
            exitp_rel=exitp/c[st+1]-1))
R=pd.DataFrame(rows).dropna(subset=["sharp"]).reset_index(drop=True)
print("events:",len(R))
print("sharpness: median %.2f  q25 %.2f  q75 %.2f"%(R.sharp.median(),R.sharp.quantile(.25),R.sharp.quantile(.75)))
THETA=R.sharp.median()
# class-conditioned: sharp(>=THETA)->time, grind(<THETA)->price
R["pnl_class"]=np.where(R.sharp>=THETA, R.pnl_time, R.pnl_price)
R["k_price_avg"]=R.kP
print("avg tranches: time %.2f  price %.2f"%(R.kT.mean(),R.kP.mean()))
print("avg entry depth vs first fill: time %.2f%%  price %.2f%% (price = lower entry)"%(R.dep_time.mean()*100,R.dep_price.mean()*100))

events: 4374
sharpness: median 0.76  q25 0.53  q75 0.89
avg tranches: time 2.81  price 1.76
avg entry depth vs first fill: time -1.08%  price -2.10% (price = lower entry)


In [3]:
def st_(a):
    a=np.array(a); return (len(a),a.mean()*100,np.median(a)*100,(a>0).mean()*100,
        np.quantile(a,.10)*100, a.mean()/a.std() if a.std()>0 else np.nan)
print("=== PER-TRADE, net (full universe -7%) ===")
print(f"{'scheme':22}{'n':>6}{'mean':>8}{'med':>8}{'win':>7}{'q10':>8}{'m/std':>8}")
for tag,col in [("A time-fixed","pnl_time"),("B price-step","pnl_price"),("C class-cond","pnl_class")]:
    n,m,md_,w,q,ms=st_(R[col])
    print(f"{tag:22}{n:>6}{m:>+8.2f}{md_:>+8.2f}{w:>6.1f}%{q:>+8.2f}{ms:>+8.3f}")
# split by class to see where price-step helps
print("\n--- by sharpness class (does price-step help the GRIND more?) ---")
for lab,mask in [("SHARP (top half)",R.sharp>=THETA),("GRIND (bottom half)",R.sharp<THETA)]:
    sub=R[mask]
    print(f"  {lab:20} n={len(sub):5d}  time {sub.pnl_time.mean()*100:+.2f}%  price {sub.pnl_price.mean()*100:+.2f}%  "
          f"(price entry {sub.dep_price.mean()*100:+.1f}% vs time {sub.dep_time.mean()*100:+.1f}%)")

=== PER-TRADE, net (full universe -7%) ===
scheme                     n    mean     med    win     q10   m/std
A time-fixed            4374   +3.82   +3.69  70.7%   -5.55  +0.375
B price-step            4374   +5.23   +4.09  71.1%   -5.30  +0.397
C class-cond            4374   +4.33   +3.86  70.8%   -5.43  +0.391

--- by sharpness class (does price-step help the GRIND more?) ---
  SHARP (top half)     n= 2187  time +5.33%  price +7.13%  (price entry -2.5% vs time -1.1%)
  GRIND (bottom half)  n= 2187  time +2.30%  price +3.32%  (price entry -1.7% vs time -1.0%)


In [4]:
# black-swan synthetic: position PnL per scheme (reuse iter08 paths)
PRE=200
def path(seg): return np.concatenate([np.ones(PRE),seg])
def lin(d,m): return 1-d*np.linspace(0,1,m)
def flat(l,m): return np.full(m,l)
scen={
 "cascade -50% (60m), no bounce": path(np.concatenate([lin(.5,60),flat(.5,300)])),
 "flash -50% (15m), no bounce":   path(np.concatenate([lin(.5,15),flat(.5,345)])),
 "flash -50% (15m), V-recover":   path(np.concatenate([lin(.5,15),.5+.45*np.linspace(0,1,120),flat(.95,225)])),
 "LUNA -90% (3h)":                path(np.concatenate([lin(.9,180),flat(.1,180)])),
}
def first_cluster(c):
    cl=clusters(c,THR); return cl[0] if cl else None
print("=== synthetic cascades: POSITION PnL per scheme (1x) ===")
print(f"{'scenario':34}{'time':>10}{'price-step':>12}")
for k,c in scen.items():
    fc=first_cluster(c)
    if fc is None: print(f"{k:34}{'NO TRIGGER':>10}"); continue
    st,en=fc; n=len(c); ex=min(en+1+H,n-1); exitp=c[ex]
    aT,_=avg_entry(c,fills_time(c,st,en),n); aP,_=avg_entry(c,fills_price(c,st,en),n)
    print(f"{k:34}{(exitp/aT-1)*100:>+9.1f}%{(exitp/aP-1)*100:>+11.1f}%")

=== synthetic cascades: POSITION PnL per scheme (1x) ===
scenario                                time  price-step
cascade -50% (60m), no bounce         -34.4%      -36.1%
flash -50% (15m), no bounce            -7.4%      -26.3%
flash -50% (15m), V-recover           +69.7%      +40.0%
LUNA -90% (3h)                        -88.0%      -87.6%


In [5]:
# portfolio (fixed $20/<=50) per scheme
PER=20.;START=1000.;MAX=50
def portfolio(pnl,entry,exit_ts):
    o=np.argsort(entry,kind="stable"); pnl=pnl[o];en=entry[o];ex=exit_ts[o]
    cash=START;openh=[];peak=START;dd=0.;worst=0.
    for i in range(len(pnl)):
        now=en[i]
        while openh and openh[0][0]<=now:
            _,pn=heapq.heappop(openh);cash+=PER*pn;peak=max(peak,cash);dd=min(dd,cash/peak-1)
        if len(openh)<MAX: heapq.heappush(openh,(ex[i],pnl[i]));worst=min(worst,pnl[i])
    for _,pn in sorted(openh):cash+=PER*pn;peak=max(peak,cash);dd=min(dd,cash/peak-1)
    span=((en.max()-en.min())/np.timedelta64(1,"D"))/365.25
    return cash/START-1,(cash/START-1)/span,dd,worst
en=R.entry.values.astype("datetime64[ns]");ex=R.exit_ts.values.astype("datetime64[ns]")
print("=== portfolio (fixed $20/<=50, full universe) ===")
for tag,col in [("A time-fixed","pnl_time"),("B price-step","pnl_price"),("C class-cond","pnl_class")]:
    tot,yr,dd,worst=portfolio(R[col].values,en,ex)
    print(f"  {tag:16} total {tot*100:+6.1f}%  yr {yr*100:+5.1f}%  maxDD {dd*100:5.1f}%  worst-trade {worst*100:6.1f}%")

=== portfolio (fixed $20/<=50, full universe) ===
  A time-fixed     total  +79.9%  yr +34.4%  maxDD  -6.2%  worst-trade  -45.3%
  B price-step     total +123.3%  yr +53.2%  maxDD  -5.5%  worst-trade  -45.3%
  C class-cond     total +100.4%  yr +43.3%  maxDD  -6.1%  worst-trade  -45.3%


## Как читать
- **Per-trade** — бьёт ли price-step фикс-время по mean/median/q10/(m/std), net.
- **by class** — где именно помогает: гипотеза iter08 = в GRIND (затяжной) price-step
  даёт более низкий вход и меньший убыток; в SHARP разницы мало.
- **Синтетика** — режет ли price-step убыток в каскаде/LUNA (ниже avg вход).
- **Портфель** — итог по году/DD/worst-trade. Если B (чистый price-step) ≥ C, то
  «классифицировать тип» не нужно — путь сам себя классифицирует.

*Verdict после прогона.*

## VERDICT (iter 10)

**Адаптивный fill (price-step) ВЫИГРЫВАЕТ.** Транш только когда close ≤
last_fill·(1−2%). Per-trade net: A time +3.82% (m/std 0.375) → B price-step
+5.23% (0.397, q10 −5.30 лучше) → C class-cond +4.33%. Портфель $20/≤50:
A +34.4%/год/DD−6.2% → **B +53.2%/год/DD−5.5%** → C +43.3%. B бьёт всех — выше
доход И ниже просадка.

**Механизм:** ниже ср.вход (−2.1% vs −1.1%), МЕНЬШЕ доборов (1.76 vs 2.81) —
избирательнее. **Путь сам себя классифицирует** → чистый price-step > class-cond,
отдельный классификатор для ТЕМПА добора не нужен (iter06 остаётся для ОТБОРА).

**Нюанс (честно): на хвосте price-step чуть ХУЖE** — при N_MAX=10/Δ=2% в очень
глубоком быстром крахе 10 траншей исчерпываются в первых −20% (грузит у верха):
flash −50% без отскока time −7.4% vs price −26.3%; V-отскок +69.7% vs +40.0%
(недобор). На реалистичном −50%/60м и LUNA разница мизерна; настоящий хвост держит
catastrophe-стоп iter09. Чистый выигрыш в норме (+19пп/год) перевешивает.

**Принято: price-step = дефолтный fill, поверх — лимиты iter09.**
**Дальше:** объединение с pump (позже), walk-forward классификатора, порт в бот.